# Terminal output 🎨

Cell output is shown the way a terminal would show it: colors and text
styles, `\r`, cursor movement and clearing the screen all work, so progress
bars, spinners and redrawing demos look right. Programs can also read
standard input.

In [ ]:
// The 256-color palette, as backgrounds.
for i := range 256 {
	fmt.Printf("\x1b[48;5;%dm  ", i)
	if i%32 == 31 {
		fmt.Println("\x1b[0m")
	}
}

In [ ]:
// hsv converts a hue, saturation and value (all 0..1) to RGB.
func hsv(h, s, v float64) (r, g, b uint8) {
	i := int(h*6) % 6
	f := h*6 - math.Floor(h*6)
	p, q, t := v*(1-s), v*(1-f*s), v*(1-(1-f)*s)
	c := [6][3]float64{{v, t, p}, {q, v, p}, {p, v, t}, {p, q, v}, {t, p, v}, {v, p, q}}[i]
	return uint8(c[0] * 255), uint8(c[1] * 255), uint8(c[2] * 255)
}

// 24-bit color: a rainbow two rows high, drawn with half blocks.
for row := range 2 {
	for x := range 64 {
		r1, g1, b1 := hsv(float64(x)/64, 0.85, 1)
		r2, g2, b2 := hsv(float64(x)/64, 0.85, 0.55+0.2*float64(1-row))
		fmt.Printf("\x1b[38;2;%d;%d;%dm\x1b[48;2;%d;%d;%dm▀", r1, g1, b1, r2, g2, b2)
	}
	fmt.Println("\x1b[0m")
}

// Text styles.
fmt.Println("\x1b[1mbold\x1b[0m \x1b[2mfaint\x1b[0m \x1b[3mitalic\x1b[0m \x1b[4munderline\x1b[0m " +
	"\x1b[7mreverse\x1b[0m \x1b[9mstrike\x1b[0m \x1b[1;38;5;208mbold orange\x1b[0m")

## Progress bars and spinners

In [ ]:
// A spinner rewrites its line with \r; \x1b[K clears what's left of it.
frames := []rune("⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏")
for i := range 40 {
	fmt.Printf("\r\x1b[K\x1b[36m%c\x1b[0m compiling the universe… %d%%", frames[i%len(frames)], i*100/39)
	time.Sleep(40 * time.Millisecond)
}
fmt.Println("\r\x1b[K\x1b[32m✓\x1b[0m compiled the universe")

In [ ]:
// downloads draws several progress bars, redrawing them in place by moving
// the cursor up over the previous frame.
func downloads(names []string, speeds []int) {
	progress := make([]int, len(names))
	for frame := 0; ; frame++ {
		if frame > 0 {
			fmt.Printf("\x1b[%dA", len(names))
		}
		done := true
		for i, name := range names {
			progress[i] = min(progress[i]+speeds[i], 100)
			done = done && progress[i] == 100
			color := 33 // yellow while running
			if progress[i] == 100 {
				color = 32
			}
			bar := strings.Repeat("█", progress[i]/4) + strings.Repeat("░", 25-progress[i]/4)
			fmt.Printf("\x1b[2K%-14s \x1b[%dm%s\x1b[0m %3d%%\n", name, color, bar, progress[i])
		}
		if done {
			return
		}
		time.Sleep(25 * time.Millisecond)
	}
}

downloads([]string{"gopher.png", "go1.27.tar.gz", "notes.txt"}, []int{3, 1, 7})

## Full-screen animation

Clearing the screen (`\x1b[2J`) starts the output over, so a program can
redraw every frame from the top.

In [ ]:
// bounce animates a ball in a box, clearing the screen for every frame.
func bounce(frames int) {
	const w, h = 36, 8
	x, y, dx, dy := 3, 2, 1, 1
	for f := range frames {
		var sb strings.Builder
		sb.WriteString("\x1b[2J\x1b[H")
		sb.WriteString("┌" + strings.Repeat("─", w) + "┐\n")
		for row := range h {
			line := []rune(strings.Repeat(" ", w))
			if row == y {
				line[x] = '●'
			}
			sb.WriteString("│\x1b[38;5;" + strconv.Itoa(196+f%36) + "m" + string(line) + "\x1b[0m│\n")
		}
		sb.WriteString("└" + strings.Repeat("─", w) + "┘\n")
		fmt.Print(sb.String())
		x, y = x+dx, y+dy
		if x <= 0 || x >= w-1 {
			dx = -dx
		}
		if y <= 0 || y >= h-1 {
			dy = -dy
		}
		time.Sleep(35 * time.Millisecond)
	}
}

bounce(70)

## Reading input

While a cell runs, an input line appears under it. Press `enter` on the
running cell (or `alt+i`, or click the line) to type; `enter` sends a line and
`ctrl+d` ends the input. With `gopyter run`, programs read gopyter's own
standard input.

In [ ]:
import (
	"bufio"
	"math/rand/v2"
)

// guessingGame reads guesses until the number is found or the input ends.
func guessingGame() {
	secret := rand.IntN(100) + 1
	fmt.Println("I'm thinking of a number between 1 and 100.")
	in := bufio.NewScanner(os.Stdin)
	for tries := 1; ; tries++ {
		fmt.Print("Your guess: ")
		if !in.Scan() {
			fmt.Println("\nGiving up? It was", secret)
			return
		}
		n, err := strconv.Atoi(strings.TrimSpace(in.Text()))
		switch {
		case err != nil:
			fmt.Println("That's not a number.")
		case n < secret:
			fmt.Println("Higher ↑")
		case n > secret:
			fmt.Println("Lower ↓")
		default:
			fmt.Printf("\x1b[32mYes! %d, in %d tries.\x1b[0m\n", secret, tries)
			return
		}
	}
}

guessingGame()

In [ ]:
// GoNB's gonbui.RequestInput focuses the input line for one line; with
// password set, what you type isn't shown or echoed.
func askPassword() string {
	gonbui.RequestInput("password", true)
	var pw string
	_, _ = fmt.Scanln(&pw) // an empty input is fine
	return pw
}

fmt.Printf("got a password of %d characters\n", len(askPassword()))